In [1]:
import polars as pl
import ast
import numpy as np
import time
import os
from tqdm import tqdm
from multiprocessing import Pool
import itertools
import matplotlib.pyplot as plt
import gc
from collections import Counter
import math
import keras
from scipy import sparse
from keras import models, layers
import keras
from keras.models import load_model
os.environ["POLARS_FMT_MAX_ROWS"] = "100"

In [2]:
def normalize_dataset(dataset: pl.DataFrame):
    bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
    dataset = dataset.with_columns(
        [
            pl.col(bool_cols).cast(pl.Float64),  # Convert boolean columns to Float64
            ((pl.col("player2_action_max_card") + 1) / 52).cast(pl.Float64),  # Normalize player2_action_max_card
            ((pl.col("player3_action_max_card") + 1) / 52).cast(pl.Float64),  # Normalize player3_action_max_card
            ((pl.col("action_max_card") + 1) / 52).cast(pl.Float64),  # Normalize action_max_card
        ]
    )
    return dataset

test_data_dir = r"E:\TLMN\data_see_opponent_card_3_players\data_test\TLMN_2025-06-09-17-34-32_CP21004ELIAS_24884_2.parquet"
test_data = pl.read_parquet(test_data_dir)
print(f"The shape of the test dataset is ({test_data.shape[0]:,}; {test_data.shape[1]})")
test_data_normalized = normalize_dataset(test_data)
MLP_model = load_model(r"models\TLMN_see_opponent_cards_3_players.keras")
X_test = test_data_normalized.select(pl.exclude("expected_return")).to_numpy()
y_pred = MLP_model.predict(X_test, verbose=1)
test_data = test_data.with_columns(pl.Series(name="y_pred", values=y_pred.flatten()))

The shape of the test dataset is (66,625; 382)
2083/2083 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step


In [3]:
# Calculate absolute difference between expected_return and predicted values
test_data = test_data.with_columns(
    pl.col("expected_return")
    .sub(pl.col("y_pred"))
    .abs()
    .alias("abs_diff")
)

print(f"\nAverage absolute difference: {test_data['abs_diff'].mean():.4f}")
print(f"Maximum absolute difference: {test_data['abs_diff'].max():.4f}")
print(f"Minimum absolute difference: {test_data['abs_diff'].min():.4f}")


Average absolute difference: 0.0480
Maximum absolute difference: 0.9991
Minimum absolute difference: 0.0000


In [ ]:
test_data["abs_diff"].describe(percentiles=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1])

In [15]:
# Get the row with maximum absolute difference
max_diff_row = test_data.filter(pl.col("abs_diff") == pl.col("abs_diff").max())

# Get the row with second maximum absolute difference
second_max_diff = test_data.select("abs_diff").unique().sort(by="abs_diff", descending=True)[1,0]
second_max_diff_row = test_data.filter(pl.col("abs_diff") == second_max_diff)

# Print the index of the row with maximum absolute difference
max_diff_idx = (test_data["abs_diff"] == second_max_diff_row["abs_diff"]).arg_max()
print(f"\nIndex of row with maximum absolute difference: {max_diff_idx}")


Index of row with maximum absolute difference: 44602


In [ ]:
from tienlen_libs import Utils, Card
# Convert dict_combo_type_to_int from key:value to value:key
dict_int_to_combo_type = {v: k for k, v in Utils.dict_combo_type_to_int.items()} | \
    {10: "Sequence 4", 11: "Sequence 5", 12: "Sequence 6", 13: "Sequence 7", 14: "Sequence 8", 19: "skip"}
print(dict_int_to_combo_type)

def print_row_info(row):
    player1_card = [i for i in range(52) if row[0, f"player1_card_{i}"] == True]
    player2_card = [i for i in range(52) if row[0, f"player2_card_{i}"] == True]
    player3_card = [i for i in range(52) if row[0, f"player3_card_{i}"] == True]
    player2_action_combo_type = [i for i in range(20) if row[0, f"player2_action_combo_type_{i}"] == True][0]
    player3_action_combo_type = [i for i in range(20) if row[0, f"player3_action_combo_type_{i}"] == True][0]
    actions_card_player1 = [i for i in range(52) if row[0, f"actions_card_player1_{i}"] == True]
    actions_card_player2 = [i for i in range(52) if row[0, f"actions_card_player2_{i}"] == True]
    actions_card_player3 = [i for i in range(52) if row[0, f"actions_card_player3_{i}"] == True]
    turn_player = [i for i in range(1, 4) if row[0, f"turn_player{i}"] == True][0]
    action_combo_type = [i for i in range(19) if row[0, f"action_combo_type_{i}"] == True][0]

    print(f"Player 1's card: ", end=""); Card.print_pretty_cards(player1_card)
    print(f"Player 2's card: ", end=""); Card.print_pretty_cards(player2_card)
    print(f"Player 3's card: ", end=""); Card.print_pretty_cards(player3_card)
    print(f"Player 2's action: {dict_int_to_combo_type[player2_action_combo_type]}", end=" "); Card.print_pretty_card(row[0, 'player2_action_max_card'])
    print(f"Player 3's action: {dict_int_to_combo_type[player3_action_combo_type]}", end=" "); Card.print_pretty_card(row[0, 'player3_action_max_card'])
    # print(f"Actions card player 1: ", end=""); Card.print_pretty_cards(actions_card_player1)
    # print(f"Actions card player 2: ", end=""); Card.print_pretty_cards(actions_card_player2)
    print(f"Action combo type: {dict_int_to_combo_type[action_combo_type]}", end=" "); Card.print_pretty_card(row[0, 'action_max_card'])
    print(f"Expected return: {row[0, 'expected_return']}, Predicted return: {row[0, 'y_pred']}")
    
# for idx in range(len(test_data)):
    # if test_data[idx, "len_player1_card"] == 1:
for idx in range(max_diff_idx - 10, max_diff_idx + 10):
    if 1:
        print(f"\nRow {idx}:")
        print_row_info(test_data[idx:idx+1])
# print_row_info(second_max_diff_row)